# 05 · AI agent walkthrough (Step 8): question → tools → report

Steps 3-7 ne bahut kuch banaya: forecast, churn, segments, anomalies, ticket NLP, document Q&A.
Manager ko 8 alag files nahi kholni — wo bas poochta hai *"West ki sales kyun gir rahi hai, kya karein?"*

**Agent = plan (kaunse tools?) → tools chalao → report (findings + matlab + actions + sources)**

| Part | File |
|---|---|
| Tools (9) | `src/agent/tools.py` |
| Planner (offline brain) | `src/agent/planner.py` |
| Report + Claude brain | `src/agent/agent.py` |
| Evaluation | `src/agent/evaluate_agent.py` |

In [1]:
import sys, os, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT); warnings.filterwarnings("ignore")
import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.max_colwidth", 120)

## 1. Tools — agent ke "haath"
Har tool ek simple Python function hai jo Steps 3-7 ke outputs padhta hai aur `{summary, data, source}` lautata hai.
**Agent khud koi number nahi banata** — har number kisi tool se aata hai, source ke saath.
`TOOL_SPECS` wahi JSON-schema format hai jo Claude jaise LLM tool-calling ke liye use karte hain.

In [2]:
from src.agent.tools import TOOLS, TOOL_SPECS
pd.DataFrame([{"tool": t["name"], "arguments": ", ".join(t["input_schema"]["properties"]) or "-",
               "what it does": t["description"]} for t in TOOL_SPECS])

,tool,arguments,what it does
0,sales_forecast,region,"Revenue forecast for the next 30 days, total or for one region, with model accuracy."
1,sales_trend,"region, category","Last quarter's revenue vs previous quarter and last year, and which categories (or regions) drove the change. Use fo..."
2,churn_risk,"region, segment, top_n","Customers likely to leave in the next 90 days: count, revenue at risk, warning signs, top customers."
3,customer_segments,-,"Customer segments (RFM clusters) with size, revenue share and recommended action."
4,ticket_insights,"category, region","Support ticket sentiment trends by category, top complaint phrases, optional region comparison."
5,urgent_calls,"region, top_n",High churn-risk customers with a recent negative ticket - who to call today.
6,order_anomalies,"region, top_n","Unusual orders flagged by the autoencoder (pricing errors, odd quantities, big discounts)."
7,customer_lookup,customer_id,"Everything about one customer: segment, revenue, churn risk, recent tickets."
8,search_documents,question,"Search company policies, SOPs and the Q3 business review PDFs; returns the relevant section with citation."


In [3]:
r = TOOLS["sales_trend"](region="West")
print(r["summary"]); pd.DataFrame(r["data"])

West revenue in Sep 2026 quarter: Rs 359.1 lakh, -8.4% vs previous quarter and -6.4% vs same quarter last year. Biggest drop: Electronics (Rs -21.9 lakh, -9.1%); biggest gain: Office Supplies (Rs 0.8 lakh). Orders 1,389 vs 1,394; active customers 444 vs 455.


,category,last_q,prev_q,change,change_pct
0,Electronics,21761059.0,23947549.5,-2186490.4,-9.1
1,Furniture,11803259.4,12765480.9,-962221.5,-7.5
2,Clothing,1116630.5,1347176.0,-230545.5,-17.1
3,Office Supplies,1229390.8,1150576.0,78814.7,6.9


## 2. Planner — kaunse tools chalane hain?
1. **normalise** — Hinglish words ko English mein ("kyun" → why, "gir" → falling)
2. **entities** — region, category, segment, customer id, top-N
3. **intents** — har tool ke keywords ka score
4. **recipes** — "why ... falling" = diagnosis (trend + churn + tickets); "kya karein" = actions (churn + urgent calls);
   "overall" = overview
5. **fallback** — kuch match nahi → documents search

In [4]:
from src.agent.planner import plan, normalise
q = "West region ki sales kyun gir rahi hai aur kya karein?"
print("normalised:", normalise(q))
p = plan(q)
print("entities:", p["entities"])
for c, why in zip(p["calls"], p["reasons"]): print(f"  {c['tool']:16s} {c['args']}   <- {why.split(': ', 1)[1]}")

normalised:  west region ki sales why falling rahi hai aur what should we do ? 
entities: {'region': 'West'}
  sales_trend      {'region': 'West'}   <- asks WHY sales changed
  churn_risk       {'region': 'West'}   <- diagnosis: are customers leaving?
  ticket_insights  {'region': 'West'}   <- diagnosis: are customers unhappy?
  urgent_calls     {'region': 'West'}   <- asks for actions -> who to call first


In [5]:
for q in ["What is the sales forecast for next month?", "Tell me about customer C01658",
          "Who approves a refund of Rs 25,000?", "top 3 customers to call today",
          "What is the SLA for a medium priority ticket?", "How is the business doing overall?"]:
    print(f"{q:48s} ->", [c["tool"] for c in plan(q)["calls"]])

What is the sales forecast for next month?       -> ['sales_forecast']
Tell me about customer C01658                    -> ['customer_lookup']
Who approves a refund of Rs 25,000?              -> ['search_documents']
top 3 customers to call today                    -> ['urgent_calls']
What is the SLA for a medium priority ticket?    -> ['search_documents']
How is the business doing overall?               -> ['sales_forecast', 'sales_trend', 'churn_risk', 'ticket_insights']


## 3. Poori report

In [6]:
from src.agent.agent import run_planner
r = run_planner("West region ki sales kyun gir rahi hai aur kya karein?", save_report=False)
display(Markdown(r["report"]))

# West region ki sales kyun gir rahi hai aur kya karein?

*Generated 24 Oct 2026 10:28 by the AI BI agent (planner).*

**How I answered:** `sales_trend(region=West)`; `churn_risk(region=West)`; `ticket_insights(region=West)`; `urgent_calls(region=West)`

## Findings

**Sales trend** - West revenue in Sep 2026 quarter: Rs 359.1 lakh, -8.4% vs previous quarter and -6.4% vs same quarter last year. Biggest drop: Electronics (Rs -21.9 lakh, -9.1%); biggest gain: Office Supplies (Rs 0.8 lakh). Orders 1,389 vs 1,394; active customers 444 vs 455.  
<sub>source: data/raw/sales.csv</sub>

**Churn risk** - West: 46 of 460 customers are High churn risk (10%), Rs 19.9 lakh annual revenue at risk. Most common warning signs: recent negative tickets (28), on Basic plan (19), orders falling (19).  
<sub>source: data/processed/customer_360.csv</sub>

**Ticket insights** - Negative share of tickets, last 3 months vs previous 9: Product Quality 36%->39% (stable); Refund 34%->36% (stable); Delivery 37%->30% (improving); Billing 39%->30% (improving); Technical 35%->24% (improving). Top complaints: Billing: debited but the order; Delivery: courier did not show; Product Quality: working within a week; Refund: refund is pending; Technical: how do i reset. In West, 23% of 2026 tickets are negative (company: 36%).  
<sub>source: reports/ticket_insights.json</sub>

**Urgent calls** - 23 high-churn-risk customers in West raised a negative ticket in the last 60 days (Rs 8.1 lakh at risk). Call them within 1 business day (Support SOP 4).  
<sub>source: data/processed/urgent_tickets.csv</sub>

## What this means

- The drop is mainly **lower spend per customer** (-6.1%), not fewer customers (-2.4% active customers) - look at basket size, prices and big-ticket orders.
- Churn risk is in line with the company average (10% vs 10%) - probably not the main cause.
- Support complaints are **not** the driver: negative tickets are lower here than overall (23% vs 36%).

## Recommended actions

1. Sales fell 8.4% last quarter, mostly in **Electronics** (-9.1%). Review Electronics pricing, stock and competitor offers with the category manager this week.
2. Routine: retention calls for High-risk customers, beginning with the biggest revenue at risk (e.g. C01430, Big-ticket buyers). SOP allows up to 10% discount or 2 months Premium free.
3. 23 high-churn-risk customers in West raised a negative ticket recently - call them within 1 business day (Support SOP section 4).
4. Negative tickets are rising in **Product Quality** (36% -> 39%); top complaint: "working within a week". Assign an owner to fix it.

<sub>Actions are rule-based suggestions from the numbers above; review before acting.</sub>

### "What this means" — numbers se nateeja
Achha analyst sirf numbers nahi batata, **comparison** karta hai — aur ye bhi batata hai ki kya wajah **nahi** hai:
- revenue change ko **customers ki ginti** aur **per-customer kharch** mein todna
- region ka churn-risk share vs company average
- region ke negative tickets vs company average

West mein: active customers sirf thode kam hue, per-customer kharch zyada gira; churn aur complaints company
average jitne ya kam → asli lever basket size / Electronics hai, support nahi.

## 4. Evaluation — naye sawalon par
Planner ke keywords banate waqt jo examples dekhe the, unpar score dikhana dhoka hai. Isliye 30 **alag**
sawal likhe (English + Hinglish), har ek ke required tools aur entities ke saath.

In [7]:
m = json.load(open("reports/agent_metrics.json"))
print({k: v for k, v in m.items() if k != "misses"})
pd.DataFrame(m["misses"])

{'questions': 30, 'tool_recall': 0.9, 'tool_precision': 0.9, 'exact_plan': 0.9, 'entity_accuracy': 1.0, 'exact_plan_english': 0.875, 'exact_plan_hinglish': 1.0, 'runs_ok': 30}


,question,required,called,entities_ok
0,What revenue do we expect over the coming month?,[sales_forecast],[search_documents],True
1,Which customers might stop buying from us?,[churn_risk],[search_documents],True
2,What kinds of customers do we have?,[customer_segments],[search_documents],True


**Honest note:** pehli baar score **87%** tha. Ek miss logic bug tha ("SLA for a ticket" policy sawal par ticket tool
bhi chal gaya) — use theek kiya → **90%**. Ye fix test dekh kar hua, isliye 90% thoda optimistic hai.
Baaki 3 miss vocabulary ke hain ("expect", "stop buying", "kinds of customers"). Unhe keyword list mein **jaan-boojh kar
nahi** joda — warna test set "yaad" ho jaata. Ye rule-based planner ki asli seema hai → yahin LLM brain kaam aata hai.

## 5. Claude brain (optional)
Same 9 tools, lekin tool choose karna Claude karta hai (tool-use loop):
```
user question → Claude: "call sales_trend(West), churn_risk(West)" → we run tools → results back → Claude writes report
```

In [8]:
from src.agent.agent import SYSTEM
print(SYSTEM)

You are a business-intelligence analyst for Kaveri Retail. Answer the user's question by calling the tools - never invent numbers; every number must come from a tool result. Call several tools when a question needs diagnosis (e.g. why sales fell -> trend, churn, tickets). Then write a short report: '## Findings' (bullets with numbers and the tool used) and '## Recommended actions' (numbered, concrete, owner + timeframe). Reply in the user's language (English or Hinglish).


In [9]:
# API key ke bina loop ki plumbing test: fake client jo real API jaisa response deta hai
from src.agent import test_claude_loop
test_claude_loop.main()

Claude tool-use loop: OK (2 API calls, 2 tools run, results returned with matching ids)


Enable karne ke liye:
```
pip install anthropic
set ANTHROPIC_API_KEY=your-key
python -m src.agent.ask_agent
```
Real API ke saath ye project mein test nahi hua — enable karne ke baad `evaluate_agent.py` jaisa test Claude brain
par bhi chalaiye (kaunse tools chune, numbers tools se match karte hain ya nahi).

## Takeaways
- Agent = **tools + planner + report**. Tools achhe hon toh agent achha hota hai — Steps 3-7 hi asli kaam hain.
- **Number kabhi invent nahi** — har line ke saath source.
- **Interpretation** (comparison, "ye wajah nahi hai") report ko useful banata hai.
- **Unseen test set** + honest misses; vocabulary ki seema → LLM brain.
- Rule-based actions sirf suggestion hain — insaan review kare.